# BioNeMo NIM Deployment Guide

Copyright (c) 2026, NVIDIA CORPORATION. Licensed under the Apache License, Version 2.0 (the "License") you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.

---

This notebook serves as a reference for deploying or configuring the BioNeMo NIM endpoints used by the bootcamp. On x86 deployment clusters, the recommended path is the Apptainer/Singularity service wrapper, which starts MolMIM and Boltz-2 and writes the endpoint environment for the notebooks. On ARM64 nodes, run Boltz-2 locally and set `MOLMIM_URL` to a hosted MolMIM endpoint.

## What This Notebook Covers

- Configure NGC access without writing secrets into notebooks.
- Prefer Apptainer/Singularity on clusters where Docker runtime is unavailable.
- Start MolMIM and one or more Boltz-2 endpoints with the repository wrapper, or configure hosted MolMIM with local Boltz-2 on ARM64.
- Verify `.openhackathon-nims.env` so downstream notebooks can discover the correct service URLs.


### Setup NGC API Key

The first step in deploying BioNeMo NIM endpoints is configuring your environment to access [NVIDIA NGC](https://catalog.ngc.nvidia.com/). We first set our `NGC_API_KEY` variable to pull the necessary NIM container images. If you don't have an NGC API key, please visit the NVIDIA NGC API [homepage](https://org.ngc.nvidia.com/setup/api-key) to generate your key. To set your key in a notebook, use the example below to either read the environment variable or accept the key as input.

```python
import os
import getpass

if not os.environ.get("NGC_API_KEY", ""):
    nvapi_key = getpass.getpass("Enter your NGC API key: ")
    os.environ["NGC_API_KEY"] = nvapi_key
```
If the key is not set in the `NGC_API_KEY` variable, you will be prompted to enter your key:
```
Enter your NGC API key:  ········
```

### Self-Hosted NIMs

The next step is checking which container runtime is available on your GPU node. On the AI-Powered-Drug-Discovery-Bootcamp deployment cluster, Docker is not expected to be available to learners, so the primary path uses Apptainer/Singularity.

```bash
command -v apptainer || command -v singularity
```

If your cluster uses environment modules, load the provided Apptainer module first:

```bash
module load apptainer/1.4.2
```

Docker workstation users can follow the alternate Docker commands in [`../deployment.md`](../deployment.md).

### Authenticate to NGC for Apptainer/Singularity

Apptainer pulls Docker/OCI images directly from `nvcr.io`. Use the special NGC username `$oauthtoken` and your NGC API key as the password. Keep the key in environment variables rather than embedding it in commands or notebooks.

```bash
export NGC_API_KEY=<PASTE_API_KEY_HERE>
export APPTAINER_DOCKER_USERNAME='\$oauthtoken'
export APPTAINER_DOCKER_PASSWORD="$NGC_API_KEY"
```

The helper scripts in this repository set `APPTAINER_DOCKER_USERNAME` and `APPTAINER_DOCKER_PASSWORD` automatically from `NGC_API_KEY` when they pull images.

### Pull NIM Images as SIF Files

The helper script pulls NIM images on first use. You can choose where images and model artifacts are cached:

```bash
export SIF_DIR=$PWD/.sif
export LOCAL_NIM_CACHE=${LOCAL_NIM_CACHE:-$HOME/.cache/nim}
mkdir -p "$SIF_DIR" "$LOCAL_NIM_CACHE"
```

MolMIM uses `nvcr.io/nim/nvidia/molmim:1.0.0` on x86. That tag is not a native ARM64 container, so ARM64 deployments should set `MOLMIM_URL` to a hosted endpoint. Boltz-2 uses `nvcr.io/nim/mit/boltz2:1.6.0` by default on ARM hosts with 590-series or newer NVIDIA drivers and CUDA 13.1 support. The Docker launcher falls back to `1.4.0` on ARM hosts with pre-590 NVIDIA drivers.

You can verify pulled Apptainer/Singularity images with:

```bash
ls -lh .sif
```

If you need to override an image tag for a specific event environment, set `MOLMIM_IMAGE` or `BOLTZ2_IMAGE` before launching:

```bash
export BOLTZ2_IMAGE=nvcr.io/nim/mit/boltz2:1.6.0
```

#### Setting up Cache for the Model Artifacts

The NIMs download a number of files for ensuring the best profiles are selected to achieve max performance on your system. To save this data and avoid downloading each time you run the container, set up a location for caching the model artifacts as `LOCAL_NIM_CACHE` and export as an environment variable.  For example,
```python
from os.path import expanduser
home = expanduser("~")
os.environ['LOCAL_NIM_CACHE']=f"{home}/.cache/nim"
```
Then make sure this directory exists and is writable by the container:
```bash
mkdir -p "$LOCAL_NIM_CACHE"
chmod 777 "$LOCAL_NIM_CACHE"
```

### Launch the Bootcamp NIM Services

For the AI-Powered-Drug-Discovery-Bootcamp deployment cluster, the simplest x86 path is to use the service wrapper. It starts MolMIM plus one or more Boltz-2 endpoints with Apptainer/Singularity, writes logs under `logs/nims/`, and generates `.openhackathon-nims.env` for the notebooks. For ARM64 Docker nodes, provide hosted MolMIM and start only local Boltz-2.

```bash
export NGC_API_KEY=<PASTE_API_KEY_HERE>
scripts/openhackathon_services.sh start --boltz2 1
source .openhackathon-nims.env
scripts/openhackathon_services.sh status
python scoring/check_dependencies.py
```

The actual selected endpoints are written to `.openhackathon-nims.env`. Source that file before running notebooks or scoring jobs. On shared nodes, ports may shift if the defaults are already in use. Start with one Boltz-2 endpoint unless you have confirmed the cluster can isolate GPUs for multiple Apptainer services.

Stop the services with:

```bash
scripts/openhackathon_services.sh stop
```


### Manual NIM Launch

If you need to debug an individual service, use the lower-level Apptainer launcher directly:

```bash
scripts/run_nim_apptainer.sh molmim 8001 0
scripts/run_nim_apptainer.sh boltz2 8000 0
# then export the matching MOLMIM_URL/BOLTZ2_URL, or prefer openhackathon_services.sh
```

For additional Boltz-2 replicas, avoid the MolMIM port and use `8010+`. Only scale out after confirming the local Apptainer GPU isolation mode:

```bash
scripts/launch_multiple_boltz2_apptainer.sh 3 8010
scripts/check_nim_health.sh boltz2 8010 3
export BOLTZ2_ENDPOINTS="http://localhost:8000,http://localhost:8010,http://localhost:8011,http://localhost:8012"
# or source .openhackathon-nims.env when using openhackathon_services.sh
```

Docker workstation and ARM64 users can follow [`../deployment.md`](../deployment.md), including the hosted-MolMIM/local-Boltz-2 launch path.

## Additional Tools
In addition to the BioNeMo NIM inference endpoints, we will leverage some additional tools to extend the NIM functionality to build end-to-end workflows in drug discovery. In addition to BioNeMo NIMs, we'll also look at ways to leverage the broader BioNeMo Platform, including open source research models published at [https://github.com/NVIDIA-Digital-Bio](https://github.com/NVIDIA-Digital-Bio).

### Boltz2-Python-Client

The Boltz-2 Python client provides a comprehensive Python client for NVIDIA's Boltz-2 biomolecular structure prediction service. This package provides both synchronous and asynchronous interfaces, a rich CLI, and built-in 3D visualization capabilities. This client will be used as part of the hands-on exercises in the following notebooks. You can find additional details at the PyPi page and on the BioNeMo examples github.

 - https://pypi.org/project/boltz2-python-client/
 - https://github.com/NVIDIA/digital-biology-examples/tree/main/examples/nims/boltz-2

### ReaSyn with MCP Server

We also include an example using our open source research models available at [https://github.com/NVIDIA-Digital-Bio](https://github.com/NVIDIA-Digital-Bio). Here we'll look at how we can package the ReaSyn model for predicting a molecule's synthesis pathway, reaction steps from building blocks to final product(s), using an encoder-decoder Transformer and a Chain-of-Reaction (CoR) notation. This includes an MCP server which exposes inference endpoints that can be used directly via python requests or exposed to an LLM agent to incorporate domain-specific skills in agentic workflows.

We'll see how this can be used in the following notebooks. ReaSyn is optional in this AI-Powered-Drug-Discovery-Bootcamp version; start a compatible MCP server and set `REASYN_URL` to enable the synthesis cells.
